# Deterministic Tractography

Tractography reconstructs white matter pathways by propagating streamlines through the diffusion model. **Deterministic** tractography always moves to the single most likely direction at each step — like following a road on a map with no uncertainty.

## Algorithm: Euler integration on the fibre orientation field

```
1. Seed a point (random or from a mask)
2. At current position p:
   a. Interpolate the fibre orientation field (FOD or DTI principal direction)
   b. Take a step of size Δs in that direction: p_new = p + Δs * v̂
3. Stop if: FA < threshold | curvature > max | length > max
4. Store streamline; go to step 1
```

## Three implementations

| Tool | Algorithm name | Input | Notes |
|---|---|---|---|
| FSL | DTIFIT + probtrackx (det. mode) | DTI principal eigenvector | Actually uses probabilistic mode; det. option available |
| MRtrix3 | SD_STREAM | WM FOD | Follows FOD peak (not DTI); handles crossings |
| DIPY | EuDX / deterministic | DTI peaks or CSD peaks | Pure Python, flexible |

> **Key insight**: MRtrix3 and DIPY can do deterministic tractography on **CSD FOD peaks**, not just DTI — this means they handle crossing fibres, which pure DTI-based methods cannot.

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../../scripts')
from utils import run

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir = Path('../../data/hcp/100307/preprocessed')
csd_dir  = Path('../../data/hcp/100307/csd')
tck_dir  = Path('../../data/hcp/100307/tractography')
tck_dir.mkdir(parents=True, exist_ok=True)

mask = str(data_dir / 'nodif_brain_mask.nii.gz')

# FOD from previous step
wm_fod_norm = str(csd_dir / 'wmfod_norm.mif')

print('Setup complete.')

## Approach A: MRtrix3 tckgen (SD_STREAM)

In [ ]:
# ─── [MRtrix3] tckgen SD_STREAM ───────────────────────────────────────────────
#
# SD_STREAM = Streamline Deterministic tractography on the Spherical Deconvolution FOD
#
# Key parameters:
#   -select N       : generate N streamlines (not seeds)
#   -seed_image     : seed randomly within the mask
#   -minlength 10   : discard very short streamlines (noise)
#   -maxlength 250  : discard implausibly long streamlines
#   -angle 45       : max turning angle per step (45° is standard)
#   -step 0.5       : step size in mm (0.5 × voxel size is typical)

det_tck_mrt = str(tck_dir / 'det_SD_STREAM_100k.tck')

mrt_det_cmd = [
    'tckgen',
    wm_fod_norm,
    det_tck_mrt,
    '-algorithm', 'SD_STREAM',
    '-seed_image', mask,
    '-select', '100000',
    '-minlength', '10',
    '-maxlength', '250',
    '-angle', '45',
    '-step', '0.5',
    '-force',
]
print('[MRtrix3] tckgen SD_STREAM command:')
print(' '.join(mrt_det_cmd))
print()

result = subprocess.run(mrt_det_cmd, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('✓ MRtrix3 deterministic tractography complete')
    # Check streamline count
    info = subprocess.run(['tckinfo', det_tck_mrt], capture_output=True, text=True)
    print(info.stdout)

## Approach B: DIPY deterministic tractography

In [ ]:
# ─── [DIPY] Deterministic tractography ────────────────────────────────────────
#
# We use CSD peaks computed in the previous chapter.

from dipy.io.gradients import read_bvals_bvecs
from dipy.core.gradients import gradient_table
from dipy.reconst.csdeconv import ConstrainedSphericalDeconvModel, auto_response_ssst
from dipy.direction import peaks_from_model, DeterministicMaximumDirectionGetter
from dipy.tracking.local_tracking import LocalTracking
from dipy.tracking.streamline import Streamlines
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion
from dipy.tracking import utils as tracking_utils
from dipy.data import get_sphere
from dipy.io.stateful_tractogram import StatefulTractogram, Space
from dipy.io.streamline import save_trk

# Load data
bvals, bvecs = read_bvals_bvecs(
    str(data_dir / 'bvals'), str(data_dir / 'bvecs'))
gtab = gradient_table(bvals, bvecs)

img   = nib.load(str(data_dir / 'data.nii.gz'))
data  = img.get_fdata()
maskd = nib.load(mask).get_fdata().astype(bool)

# Use b=1000 shell
sel   = (bvals < 50) | ((bvals > 900) & (bvals < 1100))
gtab1 = gradient_table(bvals[sel], bvecs[sel])
d1    = data[..., sel]

print('Fitting CSD model for DIPY tractography ...')
response, ratio = auto_response_ssst(gtab1, d1, roi_radii=10, fa_thr=0.7)
csd_model = ConstrainedSphericalDeconvModel(gtab1, response, sh_order=8)

sphere = get_sphere('symmetric724')
csd_peaks = peaks_from_model(
    model=csd_model, data=d1, sphere=sphere,
    relative_peak_threshold=0.5, min_separation_angle=25,
    mask=maskd, npeaks=3, normalize_peaks=True,
)
print('✓ CSD peaks computed')

In [ ]:
# Set up stopping criterion and seeds

# Stopping criterion: GFA threshold
stopping_criterion = ThresholdStoppingCriterion(csd_peaks.gfa, 0.25)

# Seeds: place one seed per voxel in the WM mask (FA > 0.2)
from dipy.reconst.dti import TensorModel, fractional_anisotropy
tenmodel = TensorModel(gtab1, fit_method='WLS')
tenfit   = tenmodel.fit(d1, mask=maskd)
FA       = fractional_anisotropy(tenfit.evals)

wm_mask = FA > 0.2
seeds   = tracking_utils.seeds_from_mask(
    wm_mask, affine=img.affine, density=1
)   # 1 seed per voxel; use density=2 for more seeds

print(f'Generated {len(seeds):,} seeds from FA > 0.2 mask')

# Direction getter: deterministic (follows peak direction)
dir_getter = DeterministicMaximumDirectionGetter.from_shcoeff(
    csd_peaks.shm_coeff,
    max_angle=45.0,
    sphere=sphere,
)

# Run tractography
print('Running DIPY deterministic tractography ...')
streamline_generator = LocalTracking(
    direction_getter=dir_getter,
    stopping_criterion=stopping_criterion,
    seeds=seeds,
    affine=img.affine,
    step_size=0.5,
    max_cross=1,
)

streamlines = Streamlines(streamline_generator)
# Filter by length
lengths = np.array([len(s) * 0.5 for s in streamlines])   # in mm
valid   = (lengths >= 10) & (lengths <= 250)
streamlines = streamlines[valid]

print(f'✓ {len(streamlines):,} streamlines (after length filtering)')

# Save as .trk
dipy_trk = str(tck_dir / 'det_dipy.trk')
sft = StatefulTractogram(streamlines, img, Space.RASMM)
save_trk(sft, dipy_trk)
print(f'Saved: {dipy_trk}')

## Approach C: FSL probtrackx2 (deterministic mode)

FSL's tractography tool is `probtrackx2`. In practice, FSL is almost always used in **probabilistic** mode (bedpostX-based), but a deterministic run can be done with `--pd` (mean direction only).

In [ ]:
# ─── [FSL] probtrackx2 ────────────────────────────────────────────────────────
#
# FSL tractography requires bedpostX output (tensor field + uncertainty samples).
# bedpostX is very slow (~8h on CPU per subject), so we show the command structure
# but do not execute it here.
#
# Step 1: Run bedpostX on the eddy-corrected data directory
# Step 2: Run probtrackx2 on the bedpostX output

bedpostx_cmd = [
    'bedpostx',
    str(prep_dir),         # directory with data.nii.gz, bvals, bvecs, nodif_brain_mask
    '--nf=3',              # 3 fibre populations per voxel
    '--fudge=1',
    '--bi_convex',
]
print('[FSL] bedpostX command (do not run — takes ~8h on CPU, ~1h on GPU):')
print(' '.join(bedpostx_cmd))
print()

# After bedpostX completes, the output is in prep_dir.bedpostX/
bedpostx_dir = str(prep_dir) + '.bedpostX'

probtrackx_cmd = [
    'probtrackx2',
    '-s', f'{bedpostx_dir}/merged',
    '-m', mask,
    '-x', mask,              # seed from whole brain
    '--dir=' + str(tck_dir / 'fsl_probtrackx'),
    '--loopcheck',
    '--onewaycondition',
    '--nsamples=5000',
    '--nsteps=2000',
    '--steplength=0.5',
    '--distthresh=10',       # min length 10 mm
]
print('[FSL] probtrackx2 command (after bedpostX):')
print(' '.join(probtrackx_cmd))
print()
print('>> FSL tractography is covered in detail in the probabilistic tractography notebook')

## Visualise the streamlines

In [ ]:
# 2D projection of streamlines (works without fury/VTK)
from dipy.io.streamline import load_trk

sft_loaded = load_trk(dipy_trk, img)
sls = sft_loaded.streamlines

# Sample 1000 streamlines for plotting
rng = np.random.default_rng(42)
idx = rng.integers(0, len(sls), size=min(1000, len(sls)))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Axial projection
for i in idx:
    pts = np.array(sls[i])
    axes[0].plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.1, linewidth=0.5)
axes[0].set_title('DIPY deterministic streamlines (axial view, 1000 samples)')
axes[0].set_aspect('equal')
axes[0].invert_yaxis()

# Coronal projection
for i in idx:
    pts = np.array(sls[i])
    axes[1].plot(pts[:, 0], pts[:, 2], 'r-', alpha=0.1, linewidth=0.5)
axes[1].set_title('DIPY deterministic streamlines (coronal view, 1000 samples)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print(f'Total streamlines: {len(sls):,}')
lengths_mm = [len(s) * 0.5 for s in sls]
print(f'Length range: {min(lengths_mm):.0f} – {max(lengths_mm):.0f} mm')
print(f'Mean length: {np.mean(lengths_mm):.0f} mm')

In [ ]:
# Convert MRtrix3 .tck to density map for comparison
if Path(det_tck_mrt).exists():
    tdi_mrt = str(tck_dir / 'det_tdi_MRtrix3.nii.gz')
    subprocess.run([
        'tckmap', det_tck_mrt, tdi_mrt,
        '-template', mask,
        '-force'
    ], capture_output=True)

    tdi_data = nib.load(tdi_mrt).get_fdata()
    z = tdi_data.shape[2] // 2

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.log1p(tdi_data[:, :, z]).T, cmap='hot', origin='lower')
    ax.set_title('MRtrix3 TDI — Track Density Image\n'
                 '(log scale; bright = many streamlines passing through)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Run MRtrix3 tckgen above to generate the .tck file.')

---

## Summary

| | MRtrix3 SD_STREAM | DIPY Deterministic | FSL probtrackx2 (det) |
|---|---|---|---|
| Input model | CSD FOD | CSD peaks | DTI (via bedpostX) |
| Crossing fibres | Yes | Yes | Limited (BEDPOSTX has 3 fibres) |
| Speed | Fast | Moderate | Very slow (bedpostX ~8h) |
| Output format | .tck | .trk | path distributions |
| Use case | Anatomical connectivity | Research, prototyping | Atlas-based ROI tracking |

> **Deterministic tractography is fast but overconfident** — it commits to one direction even where the signal is noisy. Probabilistic tractography (next chapter) samples from the uncertainty, producing more reliable path distributions.

**Next**: [Probabilistic tractography →](02_probabilistic.ipynb)